In [1]:
import zipfile

zip_path = "/content/Causal_Pricing_Week1_Dataset_Bundle.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/project_data")

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [2]:
import os

for root, dirs, files in os.walk("/content/project_data"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/project_data/team_guide/DATA_DICTIONARY.csv
/content/project_data/shared_data/member_responsibilities.csv
/content/project_data/shared_data/competitor_pricing.csv
/content/project_data/shared_data/regional_transactions.csv
/content/project_data/shared_data/region_network.csv
/content/project_data/shared_data/market_context.csv
/content/project_data/shared_data/treatment_calendar.csv


In [3]:
import pandas as pd

# Load the three datasets
transactions = pd.read_csv(
    "/content/project_data/shared_data/regional_transactions.csv"
)

context = pd.read_csv(
    "/content/project_data/shared_data/market_context.csv"
)

competitors = pd.read_csv(
    "/content/project_data/shared_data/competitor_pricing.csv"
)

print("All three datasets loaded successfully!")

print("\nTransactions:", transactions.shape)
print("Market Context:", context.shape)
print("Competitor Pricing:", competitors.shape)

All three datasets loaded successfully!

Transactions: (25249, 21)
Market Context: (1944, 13)
Competitor Pricing: (23348, 13)


In [4]:
# Display the first 5 rows of the transactions dataset
transactions.head()

,transaction_id,transaction_date,order_hour,region_id,region_name,product_id,product_name,category,customer_segment,units,...,selling_price,discount_pct,promotion_flag,inventory_level,demand_index,dynamic_price_multiplier,revenue,pricing_treatment_flag,treatment_phase,channel
0,T00000001,2026-01-01,17,R01,North Metro,P001,Everyday Essentials,Essentials,Premium,1,...,35.30,5,1,83,1.2195,1.0,35.30,0,Pre-treatment,App
1,T00000002,2026-01-01,8,R01,North Metro,P001,Everyday Essentials,Essentials,Returning,3,...,37.72,0,0,91,1.2195,1.0,113.17,0,Pre-treatment,App
2,T00000003,2026-01-01,15,R01,North Metro,P001,Everyday Essentials,Essentials,Returning,2,...,36.48,0,0,111,1.2195,1.0,72.96,0,Pre-treatment,App
3,T00000004,2026-01-01,14,R01,North Metro,P001,Everyday Essentials,Essentials,New,2,...,37.50,0,0,109,1.2195,1.0,75.00,0,Pre-treatment,App
4,T00000005,2026-01-01,10,R01,North Metro,P002,Home Care Pack,Home,Returning,1,...,59.15,0,0,143,1.2195,1.0,59.15,0,Pre-treatment,App


In [5]:
print("TRANSACTIONS COLUMNS")
print(transactions.columns.tolist())

print("\nCOMPETITOR PRICING COLUMNS")
print(competitors.columns.tolist())

print("\nCOMMON COLUMNS")
print(
    transactions.columns.intersection(
        competitors.columns
    ).tolist()
)

TRANSACTIONS COLUMNS
['transaction_id', 'transaction_date', 'order_hour', 'region_id', 'region_name', 'product_id', 'product_name', 'category', 'customer_segment', 'units', 'list_price', 'selling_price', 'discount_pct', 'promotion_flag', 'inventory_level', 'demand_index', 'dynamic_price_multiplier', 'revenue', 'pricing_treatment_flag', 'treatment_phase', 'channel']

COMPETITOR PRICING COLUMNS
['quote_id', 'quote_date', 'hour_bucket', 'region_id', 'region_name', 'product_id', 'product_name', 'competitor_id', 'competitor_price', 'competitor_discount_pct', 'competitor_stock_status', 'competitor_promo_flag', 'scrape_source']

COMMON COLUMNS
['region_id', 'region_name', 'product_id', 'product_name']


In [6]:
# Convert dates
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

competitors["quote_date"] = pd.to_datetime(
    competitors["quote_date"]
)

# Check date ranges
print("TRANSACTION DATE RANGE")
print(
    transactions["transaction_date"].min(),
    transactions["transaction_date"].max()
)

print("\nCOMPETITOR DATE RANGE")
print(
    competitors["quote_date"].min(),
    competitors["quote_date"].max()
)

# Check unique regions and products
print("\nTRANSACTION REGIONS:",
      transactions["region_id"].nunique())

print("COMPETITOR REGIONS:",
      competitors["region_id"].nunique())

print("\nTRANSACTION PRODUCTS:",
      transactions["product_id"].nunique())

print("COMPETITOR PRODUCTS:",
      competitors["product_id"].nunique())

TRANSACTION DATE RANGE
2026-01-01 00:00:00 2026-08-31 00:00:00

COMPETITOR DATE RANGE
2026-01-01 00:00:00 2026-08-31 00:00:00

TRANSACTION REGIONS: 8
COMPETITOR REGIONS: 8

TRANSACTION PRODUCTS: 6
COMPETITOR PRODUCTS: 6


In [7]:
# Prepare matching columns
transaction_keys = transactions[
    ["transaction_date", "region_id", "product_id"]
].drop_duplicates()

competitor_keys = competitors[
    ["quote_date", "region_id", "product_id"]
].drop_duplicates()

# Rename date column
competitor_keys = competitor_keys.rename(
    columns={"quote_date": "transaction_date"}
)

# Check matches
matches = transaction_keys.merge(
    competitor_keys,
    on=["transaction_date", "region_id", "product_id"],
    how="left",
    indicator=True
)

print("Total unique transaction combinations:", len(matches))

print("\nMATCHING RESULTS")
print(matches["_merge"].value_counts())

Total unique transaction combinations: 11664

MATCHING RESULTS
_merge
both          11664
left_only         0
right_only        0
Name: count, dtype: int64


In [8]:
# Calculate average competitor price

competitor_avg = competitors.groupby(
    ["quote_date", "region_id", "product_id"],
    as_index=False
).agg(
    avg_competitor_price=("competitor_price", "mean")
)

# Rename date column
competitor_avg = competitor_avg.rename(
    columns={"quote_date": "transaction_date"}
)

print("Aggregated competitor dataset shape:")
print(competitor_avg.shape)

print("\nFirst 5 rows:")
display(competitor_avg.head())

print("\nMissing average competitor prices:")
print(competitor_avg["avg_competitor_price"].isnull().sum())

Aggregated competitor dataset shape:
(11664, 4)

First 5 rows:


,transaction_date,region_id,product_id,avg_competitor_price
0,2026-01-01,R01,P001,37.310
1,2026-01-01,R01,P002,59.230
2,2026-01-01,R01,P003,79.925
3,2026-01-01,R01,P004,126.985
4,2026-01-01,R01,P005,95.755



Missing average competitor prices:
0


In [9]:
# Merge transactions with average competitor prices

merged_data = transactions.merge(
    competitor_avg,
    on=["transaction_date", "region_id", "product_id"],
    how="left",
    validate="many_to_one"
)

print("Original transactions:", transactions.shape)

print("Merged dataset:", merged_data.shape)

print("\nMissing competitor prices after merge:")
print(merged_data["avg_competitor_price"].isnull().sum())

print("\nFirst 5 rows:")
display(
    merged_data[
        ["transaction_date", "region_id",
         "product_id", "selling_price",
         "avg_competitor_price"]
    ].head()
)

Original transactions: (25249, 21)
Merged dataset: (25249, 22)

Missing competitor prices after merge:
0

First 5 rows:


,transaction_date,region_id,product_id,selling_price,avg_competitor_price
0,2026-01-01,R01,P001,35.30,37.31
1,2026-01-01,R01,P001,37.72,37.31
2,2026-01-01,R01,P001,36.48,37.31
3,2026-01-01,R01,P001,37.50,37.31
4,2026-01-01,R01,P002,59.15,59.23


In [10]:
# Create price gap feature
merged_data["price_gap"] = (
    merged_data["selling_price"]
    - merged_data["avg_competitor_price"]
)

# Create percentage price gap
merged_data["price_gap_pct"] = (
    merged_data["price_gap"]
    / merged_data["avg_competitor_price"]
) * 100

print("New dataset shape:", merged_data.shape)

print("\nPrice Gap Features:")
display(
    merged_data[
        ["selling_price",
         "avg_competitor_price",
         "price_gap",
         "price_gap_pct"]
    ].head()
)

print("\nMissing price gaps:")
print(merged_data["price_gap"].isnull().sum())

New dataset shape: (25249, 24)

Price Gap Features:


,selling_price,avg_competitor_price,price_gap,price_gap_pct
0,35.30,37.31,-2.01,-5.387296
1,37.72,37.31,0.41,1.098901
2,36.48,37.31,-0.83,-2.224605
3,37.50,37.31,0.19,0.509247
4,59.15,59.23,-0.08,-0.135067



Missing price gaps:
25


In [11]:
# Create discount amount feature
merged_data["discount_amount"] = (
    merged_data["list_price"]
    - merged_data["selling_price"]
)

print("Updated dataset shape:")
print(merged_data.shape)

print("\nDiscount Amount Feature:")
display(
    merged_data[
        ["list_price", "selling_price", "discount_amount"]
    ].head()
)

print("\nMissing discount amounts:")
print(merged_data["discount_amount"].isnull().sum())

Updated dataset shape:
(25249, 25)

Discount Amount Feature:


,list_price,selling_price,discount_amount
0,36.98,35.30,1.68
1,37.82,37.72,0.10
2,36.88,36.48,0.40
3,37.48,37.50,-0.02
4,59.01,59.15,-0.14



Missing discount amounts:
25


In [12]:
# Create revenue per unit feature
import numpy as np

merged_data["revenue_per_unit"] = (
    merged_data["revenue"]
    / merged_data["units"].replace(0, np.nan)
)

print("Updated dataset shape:")
print(merged_data.shape)

print("\nRevenue Per Unit Feature:")
display(
    merged_data[
        ["revenue", "units", "revenue_per_unit"]
    ].head()
)

print("\nMissing revenue per unit values:")
print(merged_data["revenue_per_unit"].isnull().sum())

Updated dataset shape:
(25249, 26)

Revenue Per Unit Feature:


,revenue,units,revenue_per_unit
0,35.30,1,35.300000
1,113.17,3,37.723333
2,72.96,2,36.480000
3,75.00,2,37.500000
4,59.15,1,59.150000



Missing revenue per unit values:
20


In [13]:
# Check missing values in engineered features

features = [
    "avg_competitor_price",
    "price_gap",
    "price_gap_pct",
    "discount_amount",
    "revenue_per_unit"
]

print("Missing Values in Engineered Features:")
print(merged_data[features].isnull().sum())

Missing Values in Engineered Features:
avg_competitor_price     0
price_gap               25
price_gap_pct           25
discount_amount         25
revenue_per_unit        20
dtype: int64


In [14]:
# Handle missing values in engineered features

features = [
    "avg_competitor_price",
    "price_gap",
    "price_gap_pct",
    "discount_amount",
    "revenue_per_unit"
]

# Keep original missing values unchanged
# Create indicators to identify missing data

for col in features:
    merged_data[col + "_missing"] = (
        merged_data[col].isnull().astype(int)
    )

print("Updated dataset shape:")
print(merged_data.shape)

print("\nMissing value indicators created:")
print(merged_data[
    [col + "_missing" for col in features]
].sum())

Updated dataset shape:
(25249, 31)

Missing value indicators created:
avg_competitor_price_missing     0
price_gap_missing               25
price_gap_pct_missing           25
discount_amount_missing         25
revenue_per_unit_missing        20
dtype: int64


In [15]:
# Save engineered features dataset

import os

os.makedirs("/content/outputs", exist_ok=True)

merged_data.to_csv(
    "/content/outputs/engineered_features.csv",
    index=False
)

print("Engineered features saved successfully!")
print("Dataset shape:", merged_data.shape)

Engineered features saved successfully!
Dataset shape: (25249, 31)


In [16]:
from google.colab import files

files.download("/content/outputs/engineered_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>